# ISOM 835 · Session 9, the easy way: The Trust Kit
**Suffolk University · Sawyer Business School · Fall 2026 · Prof. Hasan Arslan**

**In one sentence:** your model scores **0.945**. Before anyone trusts it, a good boss asks **five questions**. Here each question gets **one line of code**.

| | Your boss asks… | One line | Its real name |
|---|---|---|---|
| **1** | "Is that score real?" | `p.five_exams()` · `p.past_vs_future(…)` | cross-validation · a time-based test |
| **2** | "Can we make it better?" | `p.tune()` | hyperparameter tuning |
| **3** | "Why did it say that?" | `p.receipt(438)` · `p.top_clues()` | SHAP |
| **4** | "How sure is it?" | `p.calibration()` · `p.not_sure(0.90)` | calibration · conformal prediction |
| **5** | "Does it work for everyone?" | `p.by_group('market_segment')` | error analysis by slice |
| ✍️ | "Put it on one page." | `p.model_card()` | model card |

**How every step works:** a short story → **one line** → *what you should see* → **one sentence to remember**. Run the cells top to bottom with Shift + Enter. Everything takes about 2 minutes.

**Why this matters for you:** these same lines are **Milestone 3** (now due **Thu Dec 3**). At the end, you run them on your own project data.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## Step 0 · Get the kit and the hotel

The **Trust Kit** is one small file of helper functions written for this course. Each function answers one question, prints one plain-English sentence and draws one picture. Nothing is hidden: to read the code of any step, run `p.show_code('five_exams')` (there is a cell for that at the very end).

The hotel is the same as the last three weeks: two hotels in Portugal, **30,000 bookings**, and about **37% of them get canceled**. We drop the three columns that leak the answer (Session 7).

`start(...)` does five things for you. First a **data doctor** checks the columns: it removes ID columns, turns numbers stored as text into real numbers, turns dates into year/month/weekday, and **warns you about any column that leaks the answer**. Then it locks away 20% of the bookings as the **test**, prepares the columns, trains a first model (gradient boosting, Session 8), and grades it.

In [ ]:
!wget -q -O trust_kit.py https://raw.githubusercontent.com/harslan/isom-835/master/public/session-09/trust_kit.py
from trust_kit import start

hotel = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv').sample(30_000, random_state=835)
p = start(hotel, target='is_canceled', drop=['reservation_status', 'reservation_status_date', 'assigned_room_type'], yes_means='canceled')

**What you should see:** the doctor finds nothing to fix (we already dropped the leaks), then 24,000 bookings to learn from, 6,000 locked away, and a first score: **AUC 0.945**. That is the number you are about to tell your boss.

---
# Question 1 · "Is that score real?"

🎮 *Play first:* [Station 1 of the Trust Explorer](https://isom-835.vercel.app/explore/trust/#s1).

**The story.** A student takes **one** exam and scores 94.5. Is she a 94.5 student? Or did the exam happen to ask what she had studied? With one exam you cannot tell. Schools fix this with **more exams**.

Our 0.945 also comes from **one** exam: the 6,000 bookings that happened to be locked away. So we give the model **five** exams: cut the training bookings into 5 parts, and each part takes one turn as the exam while the model learns from the other four.

In [ ]:
p.five_exams()

**What you should see:** five scores between about **0.938 and 0.944**. The honest score is about **0.941, give or take 0.002**. Our 0.945 was a little lucky.

> **Remember:** report the **average and the spread**, not your luckiest exam. *Real name: 5-fold **cross-validation**.*

**The second half of the question.** Passing a driving test in an **empty parking lot** says little about **Boston traffic**. Our exams so far mix 2015, 2016 and 2017 bookings together. But the model will be used on **next year's** bookings. So: learn from 2015–2016, take the exam on 2017.

In [ ]:
p.past_vs_future('arrival_date_year', 2017)

**What you should see:** mixed together, about **0.94**. Learn the past, predict the future: about **0.88**. Same model, same amount of data, a much harder exam: new travel agents, new prices, a world that moved.

> **Remember:** choose the exam **before** the model. Ask *"when this model is in use, what will be new to it?"* and test on exactly that. This drop is bigger than anything tuning will win back.

---
# Question 2 · "Can we make it better?"

🎮 *Play first:* [Station 2](https://isom-835.vercel.app/explore/trust/#s2).

**The story.** A cake recipe has **dials**: oven temperature, baking time, sugar. Our model has dials too: how big a step it takes, how big each tree is. Real name: **hyperparameters**.

We bake **12 random recipes** and **taste** each one with 3 exams inside the training bookings. The real customers (the locked test set) taste only the winner, **once**, at the very end. (About 30 seconds.)

In [ ]:
p.tune()

**What you should see:** a table of the best recipes (the top few are almost tied), the winning dials, and the test score opened once: **0.945 → about 0.946**.

> **Remember:** **choose with cross-validation, open the test set once.** Tuning is the last few percent, not the first. Compare: the past-vs-future exam moved the score by 0.06.

---
# Question 3 · "Why did it say that?"

🎮 *Play first:* [Station 3](https://isom-835.vercel.app/explore/trust/#s3).

**The story.** The front-desk manager calls: *"Your model says booking 438 has an 84% chance of canceling. Why?"* Our model is hundreds of trees; nobody can read that. But we can print a **receipt**. A restaurant receipt starts at zero and each dish adds to the total. This receipt starts at **the average booking** and each clue adds or subtracts points until we reach this guest's score. Real name: **SHAP**. (The first receipt takes about 20 seconds; the kit installs the `shap` library if your Colab doesn't have it.)

In [ ]:
p.receipt(438)

**What you should see:** start at about **38%** (the average booking). Booked **312 days ahead**: about **+23 points**. **No special requests**: +11. **Online travel agency**: +10. **Changed the booking twice**: about **−8** (people who bother to change a booking usually plan to come). Total: about **84%**. And the guest really did cancel.

The last line gives you the sentence to say to the manager. Change `438` to any number to explain another booking.

Now **all the receipts together**: which clues move the model most, on average?

In [ ]:
p.top_clues()

**What you should see:** **country** (about 11 points) and **deposit type** (about 9) lead, then special requests, market segment, lead time and travel agent.

> ⚠️ **Careful, the most common mistake:** non-refundable bookings were canceled about **99%** of the time in this data, so the model learned "non-refundable → will cancel". That does **not** mean *"stop asking for deposits and cancellations will drop"*. These were mostly bulk bookings by tour operators. SHAP explains **the model**, not the world. To learn what a policy change does, you need an experiment (an A/B test).

> **Remember:** every prediction can come with a **receipt**: one sentence a manager can use.

---
# Question 4 · "How sure is it?"

🎮 *Play first:* [Station 4](https://isom-835.vercel.app/explore/trust/#s4).

**The story, part 1.** A weather forecaster who says "70% chance of rain" should be right about 7 days in 10 when she says it. Is our model honest like that? We group the test bookings by what the model said and check what really happened. Real name: **calibration**.

In [ ]:
p.calibration()

**What you should see:** the blue dots sit **almost on the diagonal**. When the model says 70%, about 70% really cancel. Its chances are honest, off by about 1 point. (A model can rank well and still be dishonest about its chances; that's why we check.)

**The story, part 2.** A good employee does not answer every question with total confidence. Sometimes the right answer is **"I'm not sure, somebody should look."** Here the model may answer **"will cancel"**, **"no"**, or **"not sure"**, with a promise: *the truth is inside my answer at least 90% of the time.* Real name: **conformal prediction**.

In [ ]:
p.not_sure(0.90)

**What you should see:** the promise is **kept** (about 90% of new bookings), the model commits on about **94%** of bookings and says **"not sure" on about 6%**.

That pile is gold for a business: it is the **short list for a person**. Call the guest, ask for a deposit, look closer. Try `p.not_sure(0.99)`: almost half the bookings become "not sure". More certainty costs more human work. That's a business choice, not a math one.

> **Remember:** let the model say **"not sure"**, and send that pile to a person.

---
# Question 5 · "Does it work for everyone?"

**The story.** The class average is a B+. Sounds fine, but what if one group is failing and everyone else gets A's? **An average can hide a group.** So we check the model's two kinds of mistakes **group by group**:
- **missed**: it said "no", but the guest **canceled** (an empty room);
- **false alarm**: it said "will cancel", but the guest **came** (maybe we gave their room away).

In [ ]:
p.by_group('market_segment')

**What you should see:** overall the model misses about **1 cancel in 5**. But for **Direct** bookings it misses about **6 in 10**: those guests rarely cancel, so the model almost never dares to flag one. **Online travel agency** bookings, the biggest group, get the most **false alarms** (about 1 in 7).

Try another column: `p.by_group('country')` or `p.by_group('lead_time')` (numbers are cut into four equal groups automatically).

> **Remember:** check the mistakes **for each group** that matters. You can't manage a gap you never measured.

---
# The label on the box: a model card

Every food package has a **nutrition label**. A model deserves one too: **one honest page**, written for the person who will **use** the model. The kit fills in every number you computed above. You fill in the `____` lines: they need your judgment, not code.

In [ ]:
p.model_card('Hotel cancellation risk')

**What you should see:** a filled-in card. The ⚠️ line (the past-to-future score) and the "for everyone" line are the **most valuable** lines on the page. They protect the hotel, and you, when something goes wrong.

---
# Your turn

### A · Same five questions, a different business (5 minutes)
House prices in Ames, Iowa (from Session 6). This time the model predicts a **number** (the price), so the grade is **MAE**: the average miss in dollars. The kit notices this by itself: the same lines, now in dollars.

Run the cell, then answer in one sentence each:
1. What is the honest error, give or take how much? Was the single test score lucky?
2. Write the sentence you would say to a buyer about house 0's price.
3. With a 90% promise, how wide is the honest price range? Would a buyer find that useful?
4. In which neighborhood does the model miss most?

In [ ]:
houses = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-835/master/public/data/ames_housing.csv')
h = start(houses, target='SalePrice', drop=['Id'])
h.five_exams()
h.receipt(0)
h.not_sure(0.90)
h.by_group('Neighborhood', min_size=15)

### B · Milestone 3 on your own project (now due Thu Dec 3)

Milestone 3 asks for five things. Each one is a line you already ran:

| Milestone 3 asks for… | the line |
|---|---|
| a tuned model (2–3 dials inside cross-validation) | `p.five_exams()` then `p.tune()` |
| a calibration plot (if you predict yes/no) | `p.calibration()` |
| SHAP for three cases a decision-maker would recognize | `p.three_receipts()` |
| error rates by the slices that matter | `p.by_group('a column that matters')` |
| a half-page model card | `p.model_card('my model')`, then fill in the `____` lines |

**The data doctor has your back.** Watch what happens if you forget the three leaking columns of the hotel:

In [ ]:
oops = start(hotel, target='is_canceled', yes_means='canceled')       # no drop=[...] this time

**What you should see:** the doctor turns the date column into numbers, then raises a ⚠️: `reservation_status` **alone** predicts the target almost perfectly (AUC 1.00), and the model scores a "perfect" **1.000**. The status is written down *after* the guest cancels: it is a leak (Session 7). On your project, take every ⚠️ seriously: ask *"would I know this column at the moment I predict?"* If not, add it to `drop=[...]`.

The kit also works when your data is messy: numbers stored as text, IDs, dates, blanks, a Yes/No target in words, a few hundred rows. If something can't work (for example a target with three values), it stops with a 🛑 message that tells you **exactly** what to change.

Copy the cell below, put in **your file**, **your target column**, and the **columns that leak the answer**, and remove the `#`s. If your data has a date or year column, add `p.past_vs_future('that column', a cutoff)`: it is the most honest number you can report.

In [ ]:
# Your turn · B — Milestone 3 on YOUR data. Fill in the three blanks, remove the #s, run.
# mine = pd.read_csv('YOUR FILE OR URL')
# p = start(mine, target='YOUR TARGET COLUMN', drop=['COLUMNS THAT LEAK THE ANSWER'])
# p.five_exams()
# p.tune()
# p.calibration()          # yes/no projects only
# p.three_receipts()
# p.top_clues()
# p.by_group('A COLUMN THAT MATTERS TO YOUR DECISION')
# p.model_card('YOUR MODEL NAME')

## What you learned, in five lines

1. **Is it real?** One test score is partly luck: use **five exams** and report average ± spread. Then test on the **future**, the exam that looks like the job.
2. **Can it be better?** Choose dials with cross-validation; open the test set **once**. Tuning is the last few percent.
3. **Why?** A **receipt** for every prediction. It explains the model, not the world.
4. **How sure?** Check the chances are **honest** (calibration), and let the model say **"not sure"** with a promise it keeps.
5. **For everyone?** Check the mistakes **group by group**. Then put it all on **one page**: the model card.

### Our words and their real names

| what we said | the real name |
|---|---|
| five exams | k-fold **cross-validation** |
| learn the past, test the future | **time-based split** (out-of-time validation) |
| dials · tasting the recipes | **hyperparameters** · random search with cross-validation |
| the receipt | **SHAP values** (local explanation) |
| which clues matter most | **global feature importance** (mean \|SHAP\|) |
| "when it says 70%, is it 70%?" | **calibration** |
| "will cancel" / "no" / "not sure", with a promise | **conformal prediction** (prediction sets, coverage) |
| missed · false alarm | **false negative** · **false positive** |
| a group | a **slice** |

### Optional · look inside the kit
Nothing in the kit is magic. Every function is 15–40 lines of the scikit-learn you already know. Read one:

In [ ]:
p.show_code('five_exams')        # try also: 'past_vs_future', 'tune', 'not_sure', 'by_group'